# Fix Coordinates from Gaia Archive

Reads a FITS catalog with broken/missing ra/dec, queries Gaia DR3 by source_id to get correct coordinates, and overwrites the file with fixed values.

In [1]:
import numpy as np
from astropy.io import fits
from astropy.table import Table
from astroquery.gaia import Gaia

# ========== CONFIGURATION ==========
INPUT_FILE = 'SB1Cands.2_enriched.fits'  # File to fix
# ====================================

Workaround solutions for the Gaia Archive issues following the infrastructure upgrade: https://www.cosmos.esa.int/web/gaia/news#WorkaroundArchive


In [2]:
# Load the catalog
print(f"Loading {INPUT_FILE}...")
table = Table.read(INPUT_FILE)

print(f"Columns: {table.colnames}")
print(f"Number of rows: {len(table)}")

# Check current ra/dec situation
if 'ra' in table.colnames:
    print(f"\nCurrent ra dtype: {table['ra'].dtype}")
    print(f"Current ra sample: {table['ra'][:3]}")
if 'dec' in table.colnames:
    print(f"Current dec dtype: {table['dec'].dtype}")
    print(f"Current dec sample: {table['dec'][:3]}")

Loading SB1Cands.2_enriched.fits...
Columns: ['source_id', 'ra', 'dec', 'parallax', 'parallax_error', 'Gmag', 'Teff_A23', 'logg_A23', 'MH_A23', 'EBV_W25', 'EBV_W25_err', 'A_V_W25', 'A_V_W25_err', 'dist_max_W25']
Number of rows: 1258

Current ra dtype: bool
Current ra sample:   ra 
-----
False
False
False
Current dec dtype: bool
Current dec sample:  dec 
-----
False
False
False


In [3]:
# Query Gaia for correct coordinates
source_ids = table['source_id']
print(f"Querying Gaia DR3 for {len(source_ids)} sources...")

# Query in batches
batch_size = 2000
all_results = []

for i in range(0, len(source_ids), batch_size):
    batch = source_ids[i:i+batch_size]
    id_list = ','.join(str(sid) for sid in batch)
    
    query = f"""
    SELECT source_id, ra, dec, l, b
    FROM gaiadr3.gaia_source
    WHERE source_id IN ({id_list})
    """
    
    job = Gaia.launch_job(query)
    result = job.get_results()
    all_results.append(result)
    print(f"  Batch {i//batch_size + 1}: retrieved {len(result)} sources")

# Combine results
from astropy.table import vstack
gaia_coords = vstack(all_results)
print(f"\nTotal retrieved: {len(gaia_coords)} sources")

Querying Gaia DR3 for 1258 sources...
  Batch 1: retrieved 1203 sources

Total retrieved: 1203 sources


In [4]:
# Build lookup dictionary
coord_lookup = {row['source_id']: (row['ra'], row['dec'], row['l'], row['b']) 
                for row in gaia_coords}

print(f"Lookup table has {len(coord_lookup)} entries")

# Match to our catalog
new_ra = np.zeros(len(table), dtype=np.float64)
new_dec = np.zeros(len(table), dtype=np.float64)
new_l = np.zeros(len(table), dtype=np.float64)
new_b = np.zeros(len(table), dtype=np.float64)

n_matched = 0
for i, sid in enumerate(table['source_id']):
    if sid in coord_lookup:
        new_ra[i], new_dec[i], new_l[i], new_b[i] = coord_lookup[sid]
        n_matched += 1
    else:
        new_ra[i] = new_dec[i] = new_l[i] = new_b[i] = np.nan

print(f"Matched {n_matched} / {len(table)} sources")

Lookup table has 1203 entries
Matched 1203 / 1258 sources


In [5]:
# Update or add columns
if 'ra' in table.colnames:
    table.remove_column('ra')
if 'dec' in table.colnames:
    table.remove_column('dec')

table.add_column(new_ra, name='ra', index=1)
table.add_column(new_dec, name='dec', index=2)

# Add Galactic coords if not present
if 'GLON' not in table.colnames and 'l' not in table.colnames:
    table.add_column(new_l, name='GLON', index=3)
    table.add_column(new_b, name='GLAT', index=4)
    print("Added GLON, GLAT columns")

print(f"\nUpdated columns: {table.colnames}")
print(f"\nNew ra range: {np.nanmin(new_ra):.4f} to {np.nanmax(new_ra):.4f} deg")
print(f"New dec range: {np.nanmin(new_dec):.4f} to {np.nanmax(new_dec):.4f} deg")
print(f"New l range: {np.nanmin(new_l):.4f} to {np.nanmax(new_l):.4f} deg")
print(f"New b range: {np.nanmin(new_b):.4f} to {np.nanmax(new_b):.4f} deg")

Added GLON, GLAT columns

Updated columns: ['source_id', 'ra', 'dec', 'GLON', 'GLAT', 'parallax', 'parallax_error', 'Gmag', 'Teff_A23', 'logg_A23', 'MH_A23', 'EBV_W25', 'EBV_W25_err', 'A_V_W25', 'A_V_W25_err', 'dist_max_W25']

New ra range: 0.2077 to 358.6507 deg
New dec range: -80.0647 to 82.7465 deg
New l range: 0.1165 to 359.9443 deg
New b range: -72.5976 to 80.7392 deg


In [6]:
# Save (overwrite)
table.write(INPUT_FILE, overwrite=True)
print(f"\nSaved fixed catalog to: {INPUT_FILE}")


Saved fixed catalog to: SB1Cands.2_enriched.fits


In [7]:
# Verify
print("\nVerification:")
t = Table.read(INPUT_FILE)
print(f"  ra dtype: {t['ra'].dtype}")
print(f"  ra sample: {t['ra'][:3]}")
print(f"  dec sample: {t['dec'][:3]}")


Verification:
  ra dtype: >f8
  ra sample:         ra        
------------------
161.18369878255646
 301.1468170502778
 320.6143378992251
  dec sample:        dec        
------------------
-54.73725138568436
 28.23285669772869
 54.01624596027755
